# Neural Architecture Search (NAS) Leve
## Equipe
Rayane Araújo, Júlia Júnior, Marcelo Soares, João Pedro e João Arthur.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import time
import random
import numpy as np

# Definir transformações para o MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# Carregar conjuntos de treino e teste
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Criar DataLoaders
# Dica: Para a fase de busca (AG, PBIL, etc) ser rápida, podemos usar menos épocas ou amostras menores
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False)

print(f'Tamanho do treino: {len(train_dataset)}, Teste: {len(test_dataset)}')


## 1. Espaço de Busca e Modelo (CNN Dinâmica)

In [ ]:
class DynamicCNN(nn.Module):
    def __init__(self, config, input_shape=(1, 28, 28), num_classes=10):
        super(DynamicCNN, self).__init__()
        self.config = config
        self.features = nn.Sequential()
        
        activation_layer = nn.ReLU if config['activation'] == 'ReLU' else nn.LeakyReLU
        in_channels = input_shape[0]
        
        for i in range(config['n_layers']):
            out_channels = config['filters'][i]
            self.features.add_module(f'conv_{i+1}', nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1))
            self.features.add_module(f'act_{i+1}', activation_layer())
            self.features.add_module(f'pool_{i+1}', nn.MaxPool2d(kernel_size=2, stride=2))
            in_channels = out_channels
            
        with torch.no_grad():
            dummy_input = torch.zeros(1, *input_shape)
            flatten_size = self.features(dummy_input).view(-1).shape[0]
            
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flatten_size, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

def get_num_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


## 2. Funções de Treinamento, Avaliação e Fitness

In [ ]:
def train_model(model, train_loader, epochs=1, device='cpu'):
    model.to(device)
    model.train()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    start_time = time.time()
    for epoch in range(epochs):
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            loss = criterion(model(data), target)
            loss.backward()
            optimizer.step()
    return time.time() - start_time

def evaluate_model(model, test_loader, device='cpu'):
    model.to(device)
    model.eval()
    correct = 0
    total = 0
    inference_times = []
    
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            
            start_inf = time.time()
            output = model(data)
            end_inf = time.time()
            
            inference_times.append((end_inf - start_inf) * 1000)
            
            _, predicted = torch.max(output.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
            
    return 100 * correct / total, sum(inference_times) / len(inference_times)

def calculate_fitness(accuracy, num_parameters, max_params=500000, min_acc=90.0):
    fitness = accuracy
    if num_parameters > max_params:
        fitness -= (num_parameters / max_params) * 20
    if accuracy < min_acc:
        fitness -= 20
    if num_parameters <= max_params:
        fitness += ((max_params - num_parameters) / max_params) * 2.0 
    return fitness


## 3. Metaheurística 1: Algoritmo Genético (AG)

In [ ]:
# Definição do Espaço de Busca
SEARCH_SPACE = {
    'n_layers': [1, 2, 3, 4],
    'filters': [16, 32, 64, 128],
    'activation': ['ReLU', 'LeakyReLU']
}

def generate_random_individual():
    n_layers = random.choice(SEARCH_SPACE['n_layers'])
    filters = [random.choice(SEARCH_SPACE['filters']) for _ in range(n_layers)]
    activation = random.choice(SEARCH_SPACE['activation'])
    return {'n_layers': n_layers, 'filters': filters, 'activation': activation}

def evaluate_individual(config, train_loader, test_loader, device='cpu'):
    print(f"Avaliando: {config}")
    model = DynamicCNN(config).to(device)
    num_params = get_num_parameters(model)
    
    if num_params > 500000:
        print(f"-> Rejeitado (Muitos parâmetros: {num_params})")
        return calculate_fitness(0, num_params) # Penaliza severamente
        
    train_time = train_model(model, train_loader, epochs=1, device=device)
    if train_time > 180: # Limite de 3 minutos
        print(f"-> Rejeitado (Treino excedeu limite de tempo)")
        return calculate_fitness(0, num_params)
        
    acc, inf_time = evaluate_model(model, test_loader, device)
    fitness = calculate_fitness(acc, num_params)
    print(f"-> Acurácia: {acc:.2f}%, Parâmetros: {num_params}, Fitness: {fitness:.2f}")
    
    return fitness

def crossover(parent1, parent2):
    child = {}
    child['n_layers'] = random.choice([parent1['n_layers'], parent2['n_layers']])
    
    filters = []
    for i in range(child['n_layers']):
        f1 = parent1['filters'][i] if i < parent1['n_layers'] else random.choice(SEARCH_SPACE['filters'])
        f2 = parent2['filters'][i] if i < parent2['n_layers'] else random.choice(SEARCH_SPACE['filters'])
        filters.append(random.choice([f1, f2]))
        
    child['filters'] = filters
    child['activation'] = random.choice([parent1['activation'], parent2['activation']])
    return child

def mutate(individual, mutation_rate=0.2):
    if random.random() < mutation_rate:
        individual['n_layers'] = random.choice(SEARCH_SPACE['n_layers'])
        while len(individual['filters']) < individual['n_layers']:
            individual['filters'].append(random.choice(SEARCH_SPACE['filters']))
        individual['filters'] = individual['filters'][:individual['n_layers']]
        
    for i in range(individual['n_layers']):
        if random.random() < mutation_rate:
            individual['filters'][i] = random.choice(SEARCH_SPACE['filters'])
            
    if random.random() < mutation_rate:
        individual['activation'] = random.choice(SEARCH_SPACE['activation'])
        
    return individual

def genetic_algorithm(train_loader, test_loader, pop_size=5, generations=3, device='cpu'):
    population = [generate_random_individual() for _ in range(pop_size)]
    best_overall = None
    best_fitness = -float('inf')
    
    for gen in range(generations):
        print(f"\n--- Geração {gen+1}/{generations} ---")
        
        fitness_scores = [evaluate_individual(ind, train_loader, test_loader, device) for ind in population]
        
        best_idx = np.argmax(fitness_scores)
        if fitness_scores[best_idx] > best_fitness:
            best_fitness = fitness_scores[best_idx]
            best_overall = population[best_idx]
            
        print(f"Melhor da geração {gen+1}: {population[best_idx]} (Fitness: {fitness_scores[best_idx]:.2f})")
        
        new_population = [population[best_idx]] # Elitismo
        
        while len(new_population) < pop_size:
            candidates = random.sample(list(zip(population, fitness_scores)), 2)
            parent1 = max(candidates, key=lambda x: x[1])[0]
            candidates = random.sample(list(zip(population, fitness_scores)), 2)
            parent2 = max(candidates, key=lambda x: x[1])[0]
            
            child = mutate(crossover(parent1, parent2))
            new_population.append(child)
            
        population = new_population
        
    print(f"\nBusca concluída! Melhor arquitetura: {best_overall} com fitness {best_fitness:.2f}")
    return best_overall

# Para testar, descomente o bloco abaixo e execute a célula:
# best_arch = genetic_algorithm(train_loader, test_loader, pop_size=4, generations=2, device='cpu')
